In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
pd.set_option('display.max_columns', None)

In [3]:
df=pd.read_csv('../CSV Files/Gurgaon_properties_post_feature_selection_3.csv')

In [4]:
df['floor_category'].value_counts()

floor_category
Mid Floor     1769
Low Floor      927
High Floor     796
Name: count, dtype: int64

In [5]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,facilities,floor_category
0,flat,sector 7,0.45,2,2,1,Relatively New,1000.0,No,No,unfurnished,basic,Mid Floor
1,flat,sector 3,0.50,2,2,1,Old Property,722.0,No,No,semifurnished,basic,Low Floor
2,flat,sohna road,0.40,2,2,3,New Property,661.0,No,No,unfurnished,basic,High Floor
3,flat,sector 61,1.47,2,2,2,Relatively New,1333.0,No,No,unfurnished,standard,Low Floor
4,flat,sector 92,0.70,2,2,3,Under Construction,1217.0,No,No,unfurnished,basic,Mid Floor


In [6]:
df.isnull().sum()

property_type      0
sector             0
price              0
bedRoom            0
bathroom           0
balcony            0
agePossession      0
built_up_area      0
servant room       0
store room         0
furnishing_type    0
facilities         0
floor_category     0
dtype: int64

In [7]:
df['floor_category'].value_counts()

floor_category
Mid Floor     1769
Low Floor      927
High Floor     796
Name: count, dtype: int64

In [8]:
X = df.drop(columns=['price'])
y = df['price']

In [9]:
y_transformed = np.log1p(y)

In [10]:
import category_encoders as ce
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

In [11]:
columns_to_encode = ['property_type', 'balcony', 'furnishing_type', 'agePossession','facilities', 'floor_category', 'servant room', 'store room']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough',
)

In [12]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
    subsample=0.8,
    reg_lambda=5,
    n_estimators=600,
    max_depth=8,
    learning_rate=0.03,
    gamma=0,
    colsample_bytree=0.8,
    random_state=42
))
])

In [13]:
from sklearn.model_selection import KFold, cross_val_score
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2',error_score='raise')

In [14]:
scores.mean()

0.9068715901850858

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

pipeline.fit(X_train,y_train)

y_pred = pipeline.predict(X_test)

y_pred = np.expm1(y_pred)

(mean_absolute_error(np.expm1(y_test),y_pred))

0.46703718461703847

In [16]:
from sklearn.metrics import make_scorer, mean_absolute_error
from sklearn.model_selection import cross_val_score, KFold
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y, cv=kfold, scoring=mae_scorer,error_score='raise')

In [17]:
print(f"Cross-validated MAE: {-np.mean(scores)}")

Cross-validated MAE: 0.49918984808816214


# Exporting model

In [18]:
columns_to_encode = ['property_type', 'balcony', 'furnishing_type', 'agePossession','facilities', 'floor_category', 'servant room', 'store room']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough',
)

In [19]:
final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
    subsample=0.8,
    reg_lambda=5,
    n_estimators=600,
    max_depth=8,
    learning_rate=0.03,
    gamma=0,
    colsample_bytree=0.8,
    random_state=42
))
])

In [20]:
final_pipeline.fit(X,y_transformed)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](12,)","['property_type','sector','bedRoom',...,'furnishing_type','facilities', 'floor_category']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed thro

In [21]:
X['floor_category'].value_counts()

floor_category
Mid Floor     1769
Low Floor      927
High Floor     796
Name: count, dtype: int64

In [22]:
import pickle
with open('pipeline.pkl','wb') as file:
  pickle.dump(final_pipeline,file)

In [24]:
with open('df.pkl', 'rb') as file:
    temp_df = pickle.load(file)

print(temp_df['floor_category'].unique())

['Mid Floor' 'Low Floor' 'High Floor']


In [85]:
X.columns

Index(['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'facilities', 'floor_category'],
      dtype='object')

In [86]:
X.iloc[0].values

array(['flat', 'sector 7', 2, 2, '1', 'Relatively New', 1000.0, 'No',
       'No', 'unfurnished', 'basic', 'Mid Floor'], dtype=object)

In [87]:
data = [['house', 'sector 102', 4, 3, '3+', 'New Property', 2750, 'No', 'No', 'unfurnished', 'basic', 'Low Floor']]
columns = ['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'facilities', 'floor_category']

# Convert to DataFrame
one_df = pd.DataFrame(data, columns=columns)

one_df

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,facilities,floor_category
0,house,sector 102,4,3,3+,New Property,2750,No,No,unfurnished,basic,Low Floor


In [88]:
np.expm1(pipeline.predict(one_df))

array([3.5919194], dtype=float32)

In [89]:
print(X.columns.tolist())

['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony', 'agePossession', 'built_up_area', 'servant room', 'store room', 'furnishing_type', 'facilities', 'floor_category']


In [90]:
X['floor_category'].value_counts()

floor_category
Mid Floor     1769
Low Floor      927
High Floor     796
Name: count, dtype: int64

In [91]:
sorted(df['floor_category'].unique())

['High Floor', 'Low Floor', 'Mid Floor']

In [27]:
columns = {
    'property_type': sorted(df['property_type'].unique().tolist()),
    'sector': sorted(df['sector'].unique().tolist()),
    'bedRoom': sorted(df['bedRoom'].unique().tolist()),
    'bathroom': sorted(df['bathroom'].unique().tolist()),
    'balcony': sorted(df['balcony'].unique().tolist()),
    'agePossession': sorted(df['agePossession'].unique().tolist()),
    'servant room': sorted(df['servant room'].unique().tolist()),
    'store room': sorted(df['store room'].unique().tolist()),
    'furnishing_type': sorted(df['furnishing_type'].unique().tolist()),
    'facilities': sorted(df['facilities'].unique().tolist()),
    'floor_category': sorted(df['floor_category'].unique().tolist())
}

for col, values in columns.items():
    print(f"\n{col}:")
    print(values)


property_type:
['flat', 'house']

sector:
['dwarka expressway', 'gwal pahari', 'manesar', 'sector 1', 'sector 10', 'sector 102', 'sector 103', 'sector 104', 'sector 105', 'sector 106', 'sector 107', 'sector 108', 'sector 109', 'sector 11', 'sector 110', 'sector 111', 'sector 112', 'sector 113', 'sector 12', 'sector 13', 'sector 14', 'sector 17', 'sector 2', 'sector 21', 'sector 22', 'sector 23', 'sector 24', 'sector 25', 'sector 26', 'sector 27', 'sector 28', 'sector 3', 'sector 30', 'sector 31', 'sector 33', 'sector 36', 'sector 37', 'sector 38', 'sector 39', 'sector 4', 'sector 40', 'sector 41', 'sector 43', 'sector 45', 'sector 46', 'sector 47', 'sector 48', 'sector 49', 'sector 5', 'sector 50', 'sector 51', 'sector 52', 'sector 53', 'sector 54', 'sector 55', 'sector 56', 'sector 57', 'sector 58', 'sector 59', 'sector 6', 'sector 60', 'sector 61', 'sector 62', 'sector 63', 'sector 65', 'sector 66', 'sector 67', 'sector 68', 'sector 69', 'sector 7', 'sector 70', 'sector 71', 'sector

In [28]:
X['built_up_area'].describe()

count     3492.000000
mean      1856.414696
std       1204.614128
min         33.000000
25%       1210.542604
50%       1611.000000
75%       2196.000000
max      12222.000000
Name: built_up_area, dtype: float64

In [29]:
X[X['built_up_area'] < 100]

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,facilities,floor_category
29,flat,sector 110,2,2,2,Relatively New,73.0,No,No,unfurnished,basic,Low Floor
100,flat,sector 79,2,2,3,New Property,81.0,No,No,unfurnished,standard,Mid Floor
280,flat,sector 85,2,2,1,Relatively New,97.0,No,No,unfurnished,basic,High Floor
1877,flat,sector 33,2,2,2,Under Construction,85.0,No,No,unfurnished,standard,Mid Floor
2002,flat,sector 33,2,2,3,New Property,86.0,No,No,unfurnished,standard,Low Floor
2038,flat,sector 51,1,1,0,Moderately Old,80.0,No,No,unfurnished,basic,Low Floor
2204,flat,sector 79,2,2,3,New Property,93.0,No,No,unfurnished,standard,Mid Floor
2213,flat,sector 33,2,2,0,Under Construction,85.0,No,No,unfurnished,standard,High Floor
2461,flat,sector 104,2,2,3,Relatively New,86.0,No,No,unfurnished,standard,High Floor
2570,flat,sector 33,2,2,3,Relatively New,85.0,No,No,unfurnished,standard,Mid Floor
